In [8]:
%%bash
echo "===== nvidia-smi ====="
nvidia-smi --query-gpu=name,driver_version --format=csv,noheader || true
echo
echo "===== nvcc --version (if installed) ====="
nvcc --version || echo "nvcc not found"


===== nvidia-smi =====
Tesla T4, 550.54.15

===== nvcc --version (if installed) =====
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [9]:
%%bash
GPU_NAME=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -n1 2>/dev/null || echo "Unknown")
echo "Detected GPU: '$GPU_NAME'"

# Default/fallback
SM=75

if [[ "$GPU_NAME" == *T4* ]] || [[ "$GPU_NAME" == *"Tesla T4"* ]]; then
  SM=75
elif [[ "$GPU_NAME" == *A100* ]] || [[ "$GPU_NAME" == *a100* ]]; then
  SM=80
elif [[ "$GPU_NAME" == *V100* ]] || [[ "$GPU_NAME" == *"Tesla V100"* ]]; then
  SM=70
elif [[ "$GPU_NAME" == *P100* ]] || [[ "$GPU_NAME" == *"Tesla P100"* ]]; then
  SM=60
elif [[ "$GPU_NAME" == *K80* ]] || [[ "$GPU_NAME" == *"Tesla K80"* ]]; then
  SM=37
elif [[ "$GPU_NAME" == *3090* ]] || [[ "$GPU_NAME" == *3080* ]] || [[ "$GPU_NAME" == *3070* ]]; then
  SM=86
fi

echo "Selected SM (for nvcc): sm_$SM"
echo "(If this isn't correct for your GPU, edit the 'SM' variable when compiling.)"


Detected GPU: 'Tesla T4'
Selected SM (for nvcc): sm_75
(If this isn't correct for your GPU, edit the 'SM' variable when compiling.)


In [10]:
%%bash
cat > cuda_vector_demo.cu <<'EOF'
/* cuda_vector_demo.cu - full demo */
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define N 1024

#define cudaCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}

/* Kernels */
__global__ void addKernel(const int *A, const int *B, int *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) C[i] = A[i] + B[i];
}
__global__ void squareKernel(const int *C, int *D, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) D[i] = C[i] * C[i];
}
__global__ void indexDemoKernel(int *out, int n) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    if (gid < n) out[gid] = gid;
}
__global__ void reduceSharedAtomic(const int *data, int n, int *globalSum) {
    extern __shared__ int sdata[];
    unsigned int tid = threadIdx.x;
    unsigned int idx = blockIdx.x * blockDim.x + threadIdx.x;
    sdata[tid] = (idx < n) ? data[idx] : 0;
    __syncthreads();
    for (unsigned int s = blockDim.x/2; s > 0; s >>= 1) {
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }
    if (tid == 0) atomicAdd(globalSum, sdata[0]);
}
__global__ void reduceSharedWritePartial(const int *data, int n, int *partials) {
    extern __shared__ int sdata[];
    unsigned int tid = threadIdx.x;
    unsigned int idx = blockIdx.x * blockDim.x + threadIdx.x;
    sdata[tid] = (idx < n) ? data[idx] : 0;
    __syncthreads();
    for (unsigned int s = blockDim.x/2; s > 0; s >>= 1) {
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }
    if (tid == 0) partials[blockIdx.x] = sdata[0];
}

/* Demos (same as described earlier) */
void demo_serial_default_stream() {
    printf("\n=== Demo 1: Serial execution on default stream ===\n");
    size_t bytes = N * sizeof(int);
    int *hA = (int*)malloc(bytes), *hB = (int*)malloc(bytes), *hD = (int*)malloc(bytes);
    for (int i = 0; i < N; ++i) { hA[i] = i; hB[i] = 2*i; hD[i] = -1; }
    int *dA, *dB, *dC, *dD;
    cudaCheck(cudaMalloc((void**)&dA, bytes)); cudaCheck(cudaMalloc((void**)&dB, bytes));
    cudaCheck(cudaMalloc((void**)&dC, bytes)); cudaCheck(cudaMalloc((void**)&dD, bytes));
    cudaCheck(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    cudaCheck(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    int threads = 256;
    int blocks = (N + threads - 1) / threads;
    addKernel<<<blocks, threads>>>(dA, dB, dC, N);
    cudaCheck(cudaGetLastError());
    squareKernel<<<blocks, threads>>>(dC, dD, N);
    cudaCheck(cudaGetLastError());
    cudaCheck(cudaMemcpy(hD, dD, bytes, cudaMemcpyDeviceToHost));
    printf("First 8 D values (should be 9*i^2):\n");
    for (int i = 0; i < 8; ++i) printf("D[%d] = %d\n", i, hD[i]);
    cudaCheck(cudaFree(dA)); cudaCheck(cudaFree(dB)); cudaCheck(cudaFree(dC)); cudaCheck(cudaFree(dD));
    free(hA); free(hB); free(hD);
}

void demo_streams_partition_and_overlap() {
    printf("\n\n=== Demo 2: Streams + partitioning ===\n");
    size_t bytes = N * sizeof(int);
    int *hA = (int*)malloc(bytes), *hB = (int*)malloc(bytes), *hD = (int*)malloc(bytes);
    for (int i = 0; i < N; ++i) { hA[i] = i; hB[i] = 2*i; hD[i] = -1; }
    int *dA, *dB, *dC, *dD;
    cudaCheck(cudaMalloc((void**)&dA, bytes)); cudaCheck(cudaMalloc((void**)&dB, bytes));
    cudaCheck(cudaMalloc((void**)&dC, bytes)); cudaCheck(cudaMalloc((void**)&dD, bytes));
    cudaStream_t s1, s2;
    cudaCheck(cudaStreamCreate(&s1)); cudaCheck(cudaStreamCreate(&s2));
    cudaCheck(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    cudaCheck(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    int half = N/2; int threads = 256;
    int blocks_half = (half + threads - 1) / threads;
    addKernel<<<blocks_half, threads, 0, s1>>>(dA, dB, dC, half);
    squareKernel<<<blocks_half, threads, 0, s1>>>(dC, dD, half);
    addKernel<<<blocks_half, threads, 0, s2>>>(dA + half, dB + half, dC + half, half);
    squareKernel<<<blocks_half, threads, 0, s2>>>(dC + half, dD + half, half);
    cudaCheck(cudaDeviceSynchronize());
    cudaCheck(cudaMemcpy(hD, dD, bytes, cudaMemcpyDeviceToHost));
    printf("Sample D values from both partitions:\n");
    for (int i = 0; i < 4; ++i) printf("D[%d] = %d\n", i, hD[i]);
    for (int i = half; i < half+4; ++i) printf("D[%d] = %d\n", i, hD[i]);
    cudaCheck(cudaStreamDestroy(s1)); cudaCheck(cudaStreamDestroy(s2));
    cudaCheck(cudaFree(dA)); cudaCheck(cudaFree(dB)); cudaCheck(cudaFree(dC)); cudaCheck(cudaFree(dD));
    free(hA); free(hB); free(hD);
}

void demo_streams_race_condition_and_fix() {
    printf("\n\n=== Demo 3: Race condition + fix ===\n");
    size_t bytes = N * sizeof(int);
    int *hA = (int*)malloc(bytes), *hB = (int*)malloc(bytes), *hD = (int*)malloc(bytes);
    for (int i = 0; i < N; ++i) { hA[i] = i; hB[i] = 2*i; hD[i] = -1; }
    int *dA, *dB, *dC, *dD;
    cudaCheck(cudaMalloc((void**)&dA, bytes)); cudaCheck(cudaMalloc((void**)&dB, bytes));
    cudaCheck(cudaMalloc((void**)&dC, bytes)); cudaCheck(cudaMalloc((void**)&dD, bytes));
    cudaCheck(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice)); cudaCheck(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    cudaStream_t s1, s2; cudaCheck(cudaStreamCreate(&s1)); cudaCheck(cudaStreamCreate(&s2));
    int threads = 256; int blocks = (N + threads - 1) / threads;
    // BAD:
    addKernel<<<blocks, threads, 0, s1>>>(dA, dB, dC, N);
    squareKernel<<<blocks, threads, 0, s2>>>(dC, dD, N);
    cudaCheck(cudaDeviceSynchronize());
    cudaCheck(cudaMemcpy(hD, dD, bytes, cudaMemcpyDeviceToHost));
    printf("Bad (racy) results sample:\n");
    for (int i = 0; i < 5; ++i) printf("D[%d] = %d\n", i, hD[i]);
    // FIX with events:
    cudaCheck(cudaMemset(dD, 0xFF, bytes));
    cudaEvent_t ev; cudaCheck(cudaEventCreate(&ev));
    addKernel<<<blocks, threads, 0, s1>>>(dA, dB, dC, N);
    cudaCheck(cudaEventRecord(ev, s1));
    cudaCheck(cudaStreamWaitEvent(s2, ev, 0));
    squareKernel<<<blocks, threads, 0, s2>>>(dC, dD, N);
    cudaCheck(cudaStreamSynchronize(s2));
    cudaCheck(cudaMemcpy(hD, dD, bytes, cudaMemcpyDeviceToHost));
    printf("Fixed (event-ordered) results sample:\n");
    for (int i = 0; i < 5; ++i) printf("D[%d] = %d\n", i, hD[i]);
    cudaCheck(cudaEventDestroy(ev)); cudaCheck(cudaStreamDestroy(s1)); cudaCheck(cudaStreamDestroy(s2));
    cudaCheck(cudaFree(dA)); cudaCheck(cudaFree(dB)); cudaCheck(cudaFree(dC)); cudaCheck(cudaFree(dD));
    free(hA); free(hB); free(hD);
}

void demo_synchronization_examples() {
    printf("\n\n=== Demo 4: cudaMemcpyAsync + stream sync ===\n");
    size_t bytes = N * sizeof(int);
    int *hA, *hB, *hD_pinned;
    cudaCheck(cudaHostAlloc((void**)&hA, bytes, cudaHostAllocDefault));
    cudaCheck(cudaHostAlloc((void**)&hB, bytes, cudaHostAllocDefault));
    cudaCheck(cudaHostAlloc((void**)&hD_pinned, bytes, cudaHostAllocDefault));
    for (int i = 0; i < N; ++i) { hA[i] = i; hB[i] = 2*i; hD_pinned[i] = -999; }
    int *dA, *dB, *dC, *dD;
    cudaCheck(cudaMalloc((void**)&dA, bytes)); cudaCheck(cudaMalloc((void**)&dB, bytes));
    cudaCheck(cudaMalloc((void**)&dC, bytes)); cudaCheck(cudaMalloc((void**)&dD, bytes));
    cudaCheck(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice)); cudaCheck(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    cudaStream_t s; cudaCheck(cudaStreamCreate(&s));
    int threads = 256, blocks = (N + threads - 1) / threads;
    addKernel<<<blocks, threads, 0, s>>>(dA, dB, dC, N);
    squareKernel<<<blocks, threads, 0, s>>>(dC, dD, N);
    cudaCheck(cudaMemcpyAsync(hD_pinned, dD, bytes, cudaMemcpyDeviceToHost, s));
    printf("Without synchronization: may print stale values:\n");
    for (int i = 0; i < 5; ++i) printf("hD_pinned[%d] = %d\n", i, hD_pinned[i]);
    cudaCheck(cudaStreamSynchronize(s));
    printf("After cudaStreamSynchronize (correct):\n");
    for (int i = 0; i < 5; ++i) printf("hD_pinned[%d] = %d\n", i, hD_pinned[i]);
    cudaCheck(cudaStreamDestroy(s)); cudaCheck(cudaFree(dA)); cudaCheck(cudaFree(dB)); cudaCheck(cudaFree(dC)); cudaCheck(cudaFree(dD));
    cudaCheck(cudaFreeHost(hA)); cudaCheck(cudaFreeHost(hB)); cudaCheck(cudaFreeHost(hD_pinned));
}

void demo_thread_hierarchy_mapping() {
    printf("\n\n=== Demo 5: Thread hierarchy mapping ===\n");
    size_t bytes = N * sizeof(int);
    int *dOut; cudaCheck(cudaMalloc((void**)&dOut, bytes));
    int *hOut = (int*)malloc(bytes);
    int dev; cudaCheck(cudaGetDevice(&dev));
    cudaDeviceProp prop; cudaCheck(cudaGetDeviceProperties(&prop, dev));
    if (N <= prop.maxThreadsPerBlock) {
        indexDemoKernel<<<1, N>>>(dOut, N);
        cudaCheck(cudaDeviceSynchronize());
        cudaCheck(cudaMemcpy(hOut, dOut, bytes, cudaMemcpyDeviceToHost));
        printf("Mapping for <<<1, N>>> sample:\n");
        for (int i = 0; i < 8; ++i) printf("out[%d] = %d\n", i, hOut[i]);
    } else {
        printf("Skipping <<<1, N>>> because maxThreadsPerBlock=%d < N=%d\n", prop.maxThreadsPerBlock, N);
    }
    int threads = 32; int blocks = N / threads;
    indexDemoKernel<<<blocks, threads>>>(dOut, N);
    cudaCheck(cudaDeviceSynchronize());
    cudaCheck(cudaMemcpy(hOut, dOut, bytes, cudaMemcpyDeviceToHost));
    printf("Mapping for <<<N/32, 32>>> sample:\n");
    for (int i = 0; i < 8; ++i) {
        int b = i / threads; int t = i % threads;
        printf("global index %d -> blockIdx=%d threadIdx=%d out[%d]=%d\n", i, b, t, i, hOut[i]);
    }
    cudaCheck(cudaFree(dOut)); free(hOut);
}

void demo_bonus_reduction() {
    printf("\n\n=== Bonus: reduction (shared + atomic / partials) ===\n");
    size_t bytes = N * sizeof(int);
    int *hA = (int*)malloc(bytes), *hB = (int*)malloc(bytes), *hD = (int*)malloc(bytes);
    for (int i = 0; i < N; ++i) { hA[i] = i; hB[i] = 2*i; hD[i] = 0; }
    int *dA, *dB, *dC, *dD;
    cudaCheck(cudaMalloc((void**)&dA, bytes)); cudaCheck(cudaMalloc((void**)&dB, bytes));
    cudaCheck(cudaMalloc((void**)&dC, bytes)); cudaCheck(cudaMalloc((void**)&dD, bytes));
    cudaCheck(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice)); cudaCheck(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    int threads = 256; int blocks = (N + threads - 1) / threads;
    addKernel<<<blocks, threads>>>(dA, dB, dC, N);
    squareKernel<<<blocks, threads>>>(dC, dD, N);
    int *dGlobalSum; cudaCheck(cudaMalloc((void**)&dGlobalSum, sizeof(int))); cudaCheck(cudaMemset(dGlobalSum, 0, sizeof(int)));
    size_t sbytes = threads * sizeof(int);
    reduceSharedAtomic<<<blocks, threads, sbytes>>>(dD, N, dGlobalSum);
    int hostSumAtomic = 0; cudaCheck(cudaMemcpy(&hostSumAtomic, dGlobalSum, sizeof(int), cudaMemcpyDeviceToHost));
    printf("Sum via shared-block + atomicAdd: %d\n", hostSumAtomic);
    int *dPartials; cudaCheck(cudaMalloc((void**)&dPartials, blocks * sizeof(int)));
    reduceSharedWritePartial<<<blocks, threads, sbytes>>>(dD, N, dPartials);
    int *hPartials = (int*)malloc(blocks * sizeof(int));
    cudaCheck(cudaMemcpy(hPartials, dPartials, blocks * sizeof(int), cudaMemcpyDeviceToHost));
    long long hostSum = 0; for (int i = 0; i < blocks; ++i) hostSum += hPartials[i];
    printf("Sum via block partials + host final sum: %lld\n", hostSum);
    cudaCheck(cudaMemcpy(hD, dD, bytes, cudaMemcpyDeviceToHost));
    long long cpuSum = 0; for (int i = 0; i < N; ++i) cpuSum += hD[i];
    printf("CPU direct sum of D[]: %lld\n", cpuSum);
    cudaCheck(cudaFree(dA)); cudaCheck(cudaFree(dB)); cudaCheck(cudaFree(dC)); cudaCheck(cudaFree(dD));
    cudaCheck(cudaFree(dGlobalSum)); cudaCheck(cudaFree(dPartials));
    free(hA); free(hB); free(hD); free(hPartials);
}

int main() {
    printf("CUDA Vector Ops Demo (N=%d)\n", N);
    demo_serial_default_stream();
    demo_streams_partition_and_overlap();
    demo_streams_race_condition_and_fix();
    demo_synchronization_examples();
    demo_thread_hierarchy_mapping();
    demo_bonus_reduction();
    printf("\\nAll demos completed.\\n");
    return 0;
}
EOF
echo "cuda_vector_demo.cu written (size: $(wc -c < cuda_vector_demo.cu) bytes)"


cuda_vector_demo.cu written (size: 12096 bytes)


In [11]:
%%bash
# if nvcc missing, exit early
if ! command -v nvcc >/dev/null 2>&1; then
  echo "nvcc not found. Skipping nvcc compile. Use the Numba fallback cell below."
  exit 0
fi

# prefer the detected SM if available in the environment
DETECTED_SM=${SM:-75}
SM_CAND=( ${DETECTED_SM} 80 75 70 60 86 37 )

echo "Attempting to compile with candidates: ${SM_CAND[*]}"
for sm in "${SM_CAND[@]}"; do
  echo "Trying sm_${sm} ..."
  nvcc -O2 -gencode=arch=compute_${sm},code=sm_${sm} cuda_vector_demo.cu -o cuda_vector_demo 2>&1 | tee build_sm${sm}.log
  rc=${PIPESTATUS[0]}
  if [ $rc -eq 0 ] && [ -f ./cuda_vector_demo ]; then
    echo "Compiled successfully with sm_${sm}. Running binary now..."
    ./cuda_vector_demo
    echo "Done."
    exit 0
  else
    echo "Compile for sm_${sm} failed (see build_sm${sm}.log). Trying next..."
  fi
done

echo "All attempts failed. Check build_sm*.log for details, or use the Numba fallback cell below."


Attempting to compile with candidates: 75 80 75 70 60 86 37
Trying sm_75 ...
Compiled successfully with sm_75. Running binary now...
CUDA Vector Ops Demo (N=1024)

=== Demo 1: Serial execution on default stream ===
First 8 D values (should be 9*i^2):
D[0] = 0
D[1] = 9
D[2] = 36
D[3] = 81
D[4] = 144
D[5] = 225
D[6] = 324
D[7] = 441


=== Demo 2: Streams + partitioning ===
Sample D values from both partitions:
D[0] = 0
D[1] = 9
D[2] = 36
D[3] = 81
D[512] = 2359296
D[513] = 2368521
D[514] = 2377764
D[515] = 2387025


=== Demo 3: Race condition + fix ===
Bad (racy) results sample:
D[0] = 0
D[1] = 9
D[2] = 36
D[3] = 81
D[4] = 144
Fixed (event-ordered) results sample:
D[0] = 0
D[1] = 9
D[2] = 36
D[3] = 81
D[4] = 144


=== Demo 4: cudaMemcpyAsync + stream sync ===
Without synchronization: may print stale values:
hD_pinned[0] = -999
hD_pinned[1] = -999
hD_pinned[2] = -999
hD_pinned[3] = -999
hD_pinned[4] = -999
After cudaStreamSynchronize (correct):
hD_pinned[0] = 0
hD_pinned[1] = 9
hD_pinned[